# LeetCode 2257 — Count Unguarded Cells in the Grid bằng Python

Link đề: https://leetcode.com/problems/count-unguarded-cells-in-the-grid/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def countUnguarded(self, m: int, n: int, guards: List[List[int]], walls: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng "tia nhìn" theo 4 hướng trên lưới, dừng khi gặp **tường hoặc lính gác khác**.
2. Phân tích độ phức tạp **khấu hao** (amortized): vì tia dừng ở lính gác kế tiếp, mỗi ô bị mỗi hướng quét **tối đa một lần** ⇒ tổng O(m·n) dù có nhiều lính.
3. Viết lời giải **4 lượt quét** (trái→phải, phải→trái, trên→dưới, dưới→trên) với cờ "đang được nhìn" — một mẫu quét tuyến tính rất hay gặp.
4. Phân biệt **trạng thái ô**: trống / tường / lính / đã bị canh — và chỉ đếm ô **trống chưa bị canh**.
5. Đối chiếu với vét cạn "từ mỗi ô nhìn ra 4 hướng tìm lính".

## 2. Đề bài & ví dụ

Lưới `m × n` (chỉ số từ 0). `guards[i] = [r, c]` là vị trí lính gác, `walls[j] = [r, c]` là vị trí tường.

Mỗi lính gác **nhìn thấy** mọi ô theo 4 hướng (bắc, đông, nam, tây) bắt đầu từ vị trí của mình, **trừ khi bị chặn** bởi một bức tường hoặc **một lính gác khác**. Một ô **bị canh** nếu có ít nhất một lính nhìn thấy nó.

Trả về số ô **không bị chiếm** (không phải tường, không phải lính) và **không bị canh**.

**Ví dụ 1:** `m = 4, n = 6, guards = [[0,0],[1,1],[2,3]], walls = [[0,1],[2,2],[1,4]]` → `7`.

**Ví dụ 2:** `m = 3, n = 3, guards = [[1,1]], walls = [[0,1],[1,0],[2,1],[1,2]]` → `4` (lính bị bốn bức tường bao quanh, bốn góc không bị canh).

**Ràng buộc:** `1 ≤ m, n ≤ 10⁵`, `2 ≤ m·n ≤ 10⁵`, `1 ≤ len(guards), len(walls) ≤ 5·10⁴`, `len(guards) + len(walls) ≤ m·n`, các vị trí phân biệt.

In [1]:
from typing import List
import random, time

examples = [
    (4, 6, [[0, 0], [1, 1], [2, 3]], [[0, 1], [2, 2], [1, 4]], 7),
    (3, 3, [[1, 1]], [[0, 1], [1, 0], [2, 1], [1, 2]], 4),
]
EMPTY, WALL, GUARD, SEEN = 0, 1, 2, 3
def render(m, n, guards, walls, seen=None):
    g = [["." for _ in range(n)] for _ in range(m)]
    if seen:
        for r, c in seen: g[r][c] = "o"
    for r, c in walls: g[r][c] = "W"
    for r, c in guards: g[r][c] = "G"
    for row in g: print("   " + " ".join(row))

for m, n, gs, ws, exp in examples:
    print(f"m={m}, n={n}, kỳ vọng {exp}"); render(m, n, gs, ws)

m=4, n=6, kỳ vọng 7
   G W . . . .
   . G . . W .
   . . W G . .
   . . . . . .
m=3, n=3, kỳ vọng 4
   . W .
   W G W
   . W .


## 3. Kiến thức nền

### 3.1 Mã hóa trạng thái ô bằng số nguyên

Dùng một lưới `grid[r][c]` với hằng số `EMPTY = 0`, `WALL = 1`, `GUARD = 2`, `SEEN = 3`. Lính và tường là **vật cản**; ô `SEEN` vẫn cho tia đi qua (một ô đã bị canh không chặn tầm nhìn).

In [2]:
m, n = 3, 4
grid = [[EMPTY] * n for _ in range(m)]
grid[0][1] = WALL; grid[1][2] = GUARD
for row in grid: print(row)

[0, 1, 0, 0]
[0, 0, 2, 0]
[0, 0, 0, 0]


### 3.2 Đi theo một hướng tới khi gặp vật cản

In [3]:
def walk(grid, r, c, dr, dc):
    m, n = len(grid), len(grid[0]); out = []
    r, c = r + dr, c + dc
    while 0 <= r < m and 0 <= c < n and grid[r][c] not in (WALL, GUARD):
        out.append((r, c)); r, c = r + dr, c + dc
    return out

print("lính (1,2) nhìn sang trái:", walk(grid, 1, 2, 0, -1))
print("lính (1,2) nhìn lên:", walk(grid, 1, 2, -1, 0))

lính (1,2) nhìn sang trái: [(1, 1), (1, 0)]
lính (1,2) nhìn lên: [(0, 2)]


### 3.3 Quét một hàng với cờ "đang nhìn"

Đi từ trái sang phải: gặp lính ⇒ bật cờ; gặp tường ⇒ tắt cờ; ô trống khi cờ bật ⇒ bị canh. Đây là cách xử lý **mọi** lính trên hàng theo một hướng chỉ trong **một** lượt.

In [4]:
row = ["G", ".", ".", "W", ".", "G", ".", "."]
flag = False; seen = []
for j, ch in enumerate(row):
    if ch == "G": flag = True
    elif ch == "W": flag = False
    elif flag: seen.append(j)
print("ô bị canh khi nhìn sang phải:", seen)

ô bị canh khi nhìn sang phải: [1, 2, 6, 7]


### 3.4 Lưới phẳng 1 chiều (tùy chọn tối ưu)

Vì `m·n ≤ 10⁵` nhưng có thể `m = 1, n = 10⁵` (hoặc ngược lại), lưu lưới dạng list 2 chiều vẫn ổn. Một số lời giải dùng mảng 1 chiều `idx = r·n + c` để nhanh hơn chút.

## 4. Ý tưởng nền tảng / trực giác

Hãy tưởng tượng mỗi lính chiếu 4 tia đèn pin. Tia dừng khi đập vào tường hoặc vào một lính khác. Mọi ô trống bị tia chiếu tới là bị canh.

Làm tay ví dụ 2: lính ở `(1, 1)`, bốn ô kề là tường ⇒ 4 tia đều bị chặn ngay ⇒ không ô nào bị canh. Ô trống còn lại là 4 góc ⇒ đáp án 4.

Làm tay ví dụ 1 (`G` lính, `W` tường, `o` bị canh, `.` không bị canh):

In [5]:
def seen_cells(m, n, guards, walls):
    grid = [[EMPTY] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = WALL
    for r, c in guards: grid[r][c] = GUARD
    s = set()
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            s.update(walk(grid, r, c, dr, dc))
    return s

s = seen_cells(*examples[0][:4]); render(*examples[0][:4], seen=s)
print("số ô trống không bị canh:", 4 * 6 - 3 - 3 - len(s))

   G W . o . .
   o G o o W .
   o o W G o o
   o o . o . .
số ô trống không bị canh: 7


**Câu hỏi then chốt về độ phức tạp:** nếu có rất nhiều lính, mỗi lính đi tới O(m + n) ô, liệu tổng có thành O(số lính · (m + n))? Không — vì tia **dừng ở lính kế tiếp**. Trên một hàng, các lính chia hàng thành các đoạn; mỗi đoạn chỉ bị quét bởi lính ở hai đầu (một lính nhìn sang phải, một lính nhìn sang trái). ⇒ mỗi ô bị quét ≤ 2 lần theo phương ngang và ≤ 2 lần theo phương dọc ⇒ tổng **O(m·n)**.

## 5. Cách 1 — Brute Force / baseline: từ mỗi ô trống, nhìn ra 4 hướng tìm lính

**Ý tưởng:** ô trống `(r, c)` bị canh ⇔ theo một trong 4 hướng, vật cản **đầu tiên** gặp phải là một lính (không phải tường, không phải biên).

In [6]:
def brute(m, n, guards, walls):
    grid = [[EMPTY] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = WALL
    for r, c in guards: grid[r][c] = GUARD
    cnt = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] != EMPTY: continue
            guarded = False
            for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
                x, y = r + dr, c + dc
                while 0 <= x < m and 0 <= y < n and grid[x][y] == EMPTY:
                    x, y = x + dr, y + dc
                if 0 <= x < m and 0 <= y < n and grid[x][y] == GUARD:
                    guarded = True; break
            cnt += not guarded
    return cnt

for m, n, gs, ws, exp in examples:
    got = brute(m, n, gs, ws)
    print(got, "OK" if got == exp else "SAI")

7 OK
4 OK


**Độ phức tạp:** mỗi ô trống quét tới O(m + n) ⇒ **O(m·n·(m + n))**. Với lưới 316 × 316 gần như trống: ~6·10⁷ bước; lưới 1 × 10⁵ trống: 10¹⁰ bước (mỗi ô quét cả hàng) ⇒ TLE.

**Nút thắt:** các ô trống cạnh nhau trên cùng một đoạn hàng cho **cùng một câu trả lời** theo phương ngang (cùng vật cản trái/phải), nhưng ta tính lại cho từng ô.

In [7]:
for n in [2000, 8000]:
    t = time.perf_counter(); brute(1, n, [[0, 0]], [[0, n - 1]]); print(f"1 x {n}: {(time.perf_counter()-t)*1000:.0f} ms")

1 x 2000: 366 ms


1 x 8000: 5392 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (mô phỏng từ lính):** đảo chiều — thay vì mỗi ô đi tìm lính, mỗi lính chiếu tia. Tia dừng ở tường/lính ⇒ khấu hao O(m·n) (mục 4).

**Insight 2 (4 lượt quét):** xử lý mọi lính trên một hàng cùng lúc bằng cờ:
- Quét trái→phải: `flag = True` khi gặp lính, `False` khi gặp tường; ô trống gặp khi `flag` ⇒ đánh dấu.
- Tương tự phải→trái, trên→dưới, dưới→trên.

Quy tắc "nếu… thì…" trong một lượt quét:
- **Nếu** ô là lính **thì** `flag = True` (tia mới bắt đầu, và lính này chặn tia của lính phía trước — nhưng tia mới thay thế nên cờ vẫn bật).
- **Nếu** ô là tường **thì** `flag = False`.
- **Nếu** ô trống/đã canh **và** `flag` **thì** đánh dấu bị canh.

**Bất biến của một lượt quét trái→phải trên hàng r:** khi tới cột `c`, `flag = True` ⇔ vật cản gần nhất bên trái `c` (trên hàng r) là một lính.

**Lưu ý tinh tế:** "bị chặn bởi lính khác" không làm thay đổi tập ô bị canh! Nếu lính A nhìn sang phải và gặp lính B, thì các ô sau B vẫn bị B canh (B cũng nhìn sang phải). Vậy điều kiện "lính chặn tia" chỉ giúp giảm số bước, không đổi đáp án — Lỗi 4 ở mục 13 kiểm chứng điều này.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tia từ mỗi lính (cách doocs)

In [8]:
def simulate(m, n, guards, walls):
    grid = [[EMPTY] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = WALL
    for r, c in guards: grid[r][c] = GUARD
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and grid[x][y] != WALL and grid[x][y] != GUARD:
                grid[x][y] = SEEN
                x, y = x + dr, y + dc
    return sum(row.count(EMPTY) for row in grid)

for m, n, gs, ws, exp in examples:
    got = simulate(m, n, gs, ws)
    print(got, "OK" if got == exp else "SAI")

7 OK
4 OK


**Độ phức tạp cách 2:** O(m·n + số lính + số tường) nhờ khấu hao; bộ nhớ O(m·n).

Lưu ý: tia **đi xuyên** ô đã `SEEN` (không dừng), vì ô bị lính khác canh không phải vật cản.

### 7.2 Cách 3 — Bốn lượt quét với cờ

In [9]:
def four_sweeps(m, n, guards, walls):
    grid = [[EMPTY] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = WALL
    for r, c in guards: grid[r][c] = GUARD
    seen = [[False] * n for _ in range(m)]
    def sweep(cells):
        flag = False
        for r, c in cells:
            v = grid[r][c]
            if v == GUARD: flag = True
            elif v == WALL: flag = False
            elif flag: seen[r][c] = True
    for r in range(m):
        sweep([(r, c) for c in range(n)]); sweep([(r, c) for c in range(n - 1, -1, -1)])
    for c in range(n):
        sweep([(r, c) for r in range(m)]); sweep([(r, c) for r in range(m - 1, -1, -1)])
    return sum(1 for r in range(m) for c in range(n) if grid[r][c] == EMPTY and not seen[r][c])

for m, n, gs, ws, exp in examples:
    got = four_sweeps(m, n, gs, ws)
    print(got, "OK" if got == exp else "SAI")

7 OK
4 OK


**Độ phức tạp cách 3:** đúng `4·m·n` bước (không phụ thuộc số lính), bộ nhớ O(m·n). Không có vòng `while` lồng — dễ phân tích.

### 7.3 Cách 4 — Hai lượt mỗi phương bằng "vật cản gần nhất"

Với mỗi ô, tính loại vật cản gần nhất bên trái (`L`) và bên phải (`R`) bằng hai lượt quét hàng; tương tự trên (`U`) và dưới (`D`). Ô trống bị canh ⇔ một trong bốn vật cản gần nhất là lính. Cách này tách "tính thông tin" và "quyết định", dễ mở rộng (biến thể 15.2).

In [10]:
def nearest_blockers(m, n, guards, walls):
    grid = [[EMPTY] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = WALL
    for r, c in guards: grid[r][c] = GUARD
    NONE = 0
    L = [[NONE] * n for _ in range(m)]; Rr = [[NONE] * n for _ in range(m)]
    U = [[NONE] * n for _ in range(m)]; D = [[NONE] * n for _ in range(m)]
    for r in range(m):
        last = NONE
        for c in range(n):
            L[r][c] = last
            if grid[r][c] in (WALL, GUARD): last = grid[r][c]
        last = NONE
        for c in range(n - 1, -1, -1):
            Rr[r][c] = last
            if grid[r][c] in (WALL, GUARD): last = grid[r][c]
    for c in range(n):
        last = NONE
        for r in range(m):
            U[r][c] = last
            if grid[r][c] in (WALL, GUARD): last = grid[r][c]
        last = NONE
        for r in range(m - 1, -1, -1):
            D[r][c] = last
            if grid[r][c] in (WALL, GUARD): last = grid[r][c]
    return sum(1 for r in range(m) for c in range(n)
               if grid[r][c] == EMPTY and GUARD not in (L[r][c], Rr[r][c], U[r][c], D[r][c]))

for m, n, gs, ws, exp in examples:
    got = nearest_blockers(m, n, gs, ws)
    print(got, "OK" if got == exp else "SAI")

7 OK
4 OK


**Độ phức tạp cách 4:** O(m·n), bộ nhớ O(m·n) (bốn mảng phụ).

### 7.4 Cách 5 — Lưới phẳng 1 chiều + mô phỏng (tối ưu hằng số)

Dùng `bytearray` dài `m·n` và chỉ số `r·n + c`; bước ngang là ±1 (dừng ở biên hàng), bước dọc là ±n.

In [11]:
def flat_simulate(m, n, guards, walls):
    g = bytearray(m * n)                   # 0 trống, 1 tường, 2 lính, 3 bị canh
    for r, c in walls: g[r * n + c] = 1
    for r, c in guards: g[r * n + c] = 2
    for r, c in guards:
        base = r * n
        j = c + 1
        while j < n and (g[base + j] == 0 or g[base + j] == 3):
            g[base + j] = 3; j += 1
        j = c - 1
        while j >= 0 and (g[base + j] == 0 or g[base + j] == 3):
            g[base + j] = 3; j -= 1
        i = r + 1
        while i < m and (g[i * n + c] == 0 or g[i * n + c] == 3):
            g[i * n + c] = 3; i += 1
        i = r - 1
        while i >= 0 and (g[i * n + c] == 0 or g[i * n + c] == 3):
            g[i * n + c] = 3; i -= 1
    return g.count(0)

for m, n, gs, ws, exp in examples:
    got = flat_simulate(m, n, gs, ws)
    print(got, "OK" if got == exp else "SAI")

7 OK
4 OK


`bytearray.count(0)` đếm trong C ⇒ rất nhanh. Cả bốn vòng `while` dùng cùng điều kiện "ô trống hoặc đã bị canh" (tức không phải tường/lính) — tia đi xuyên ô đã bị canh.

## 8. TRACE từng bước

### 8.1 Trace cách 2 (mô phỏng tia) trên ví dụ 1 — in lưới sau mỗi lính

In [12]:
def trace_simulate(m, n, guards, walls):
    grid = [[EMPTY] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = WALL
    for r, c in guards: grid[r][c] = GUARD
    sym = {EMPTY: ".", WALL: "W", GUARD: "G", SEEN: "o"}
    for r, c in guards:
        steps = 0
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and grid[x][y] not in (WALL, GUARD):
                grid[x][y] = SEEN; x, y = x + dr, y + dc; steps += 1
        print(f"sau lính {(r, c)} ({steps} bước):")
        for row in grid: print("   " + " ".join(sym[v] for v in row))
    ans = sum(row.count(EMPTY) for row in grid)
    print("ô trống không bị canh:", ans); return ans

trace_simulate(*examples[0][:4])

sau lính (0, 0) (3 bước):
   G W . . . .
   o G . . W .
   o . W G . .
   o . . . . .
sau lính (1, 1) (5 bước):
   G W . . . .
   o G o o W .
   o o W G . .
   o o . . . .
sau lính (2, 3) (5 bước):
   G W . o . .
   o G o o W .
   o o W G o o
   o o . o . .
ô trống không bị canh: 7


7

### 8.2 Trace một lượt quét (cách 3) trên hàng 1 của ví dụ 1

Hàng 1: cột `0` trống, `1` lính, `2, 3` trống, `4` tường, `5` trống.

| cột | ô | cờ trước | hành động | cờ sau |
|---|---|---|---|---|
| 0 | . | F | không đánh dấu | F |
| 1 | G | F | bật cờ | T |
| 2 | . | T | đánh dấu | T |
| 3 | . | T | đánh dấu | T |
| 4 | W | T | tắt cờ | F |
| 5 | . | F | không đánh dấu | F |

In [13]:
def trace_sweep(row_cells):
    flag = False
    for c, ch in enumerate(row_cells):
        before = flag
        if ch == "G": flag = True; act = "bật cờ"
        elif ch == "W": flag = False; act = "tắt cờ"
        else: act = "ĐÁNH DẤU" if flag else "-"
        print(f"cột {c}: '{ch}' cờ {before!s:<5} -> {act:<9} cờ {flag}")

trace_sweep([".", "G", ".", ".", "W", "."])

cột 0: '.' cờ False -> -         cờ False
cột 1: 'G' cờ False -> bật cờ    cờ True
cột 2: '.' cờ True  -> ĐÁNH DẤU  cờ True
cột 3: '.' cờ True  -> ĐÁNH DẤU  cờ True
cột 4: 'W' cờ True  -> tắt cờ    cờ False
cột 5: '.' cờ False -> -         cờ False


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def countUnguarded(self, m: int, n: int, guards: List[List[int]], walls: List[List[int]]) -> int:
        # 0 = trống, 1 = tường, 2 = lính, 3 = bị canh
        g = [[0] * n for _ in range(m)]
        for r, c in walls:
            g[r][c] = 1
        for r, c in guards:
            g[r][c] = 2
        for r, c in guards:
            for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
                x, y = r + dr, c + dc
                while 0 <= x < m and 0 <= y < n and g[x][y] != 1 and g[x][y] != 2:
                    g[x][y] = 3                  # ô bị canh vẫn cho tia đi qua
                    x += dr
                    y += dc
        return sum(row.count(0) for row in g)

In [15]:
sol = Solution()
for m, n, gs, ws, exp in examples:
    got = sol.countUnguarded(m, n, gs, ws)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

7 | kỳ vọng 7 | OK
4 | kỳ vọng 4 | OK


## 10. Vì sao thuật toán đúng

**Tập ô bị canh.** Ô trống `X` bị canh ⇔ có lính `G` cùng hàng (hoặc cột) với `X` sao cho giữa `G` và `X` không có tường hay lính nào. Thuật toán mô phỏng: từ mỗi lính, theo mỗi hướng, đánh dấu liên tiếp các ô cho tới vật cản đầu tiên — đúng bằng tập các ô trống mà lính đó nhìn thấy. Hợp trên mọi lính, mọi hướng = tập ô bị canh. Ô `SEEN` không bị coi là vật cản nên không làm đứt tia (đúng với đề: chỉ tường và lính chặn).

**Khấu hao O(m·n).** Xét hướng "sang phải" trên hàng `r`. Một ô `(r, c)` được tia sang phải của lính `G` đi qua ⇔ `G` là **vật cản gần nhất bên trái** của `c` và là lính. Vật cản gần nhất là duy nhất ⇒ mỗi ô bị tia sang phải đi qua tối đa **một** lần. Tương tự 3 hướng còn lại ⇒ tổng ≤ `4·m·n` bước. ∎

**Cách 3 đúng** vì bất biến của cờ (mục 6) chính là "vật cản gần nhất phía trước là lính", trùng với điều kiện ở trên.

## 11. Phân tích Time / Space Complexity

`N = m·n ≤ 10⁵`, `G` = số lính, `W` = số tường.

| Cách | Time | Space |
|---|---|---|
| 1. Từ ô nhìn ra | O(N·(m+n)) | O(N) |
| 2. Mô phỏng tia | O(N + G + W) (khấu hao) | O(N) |
| 3. Bốn lượt quét | O(N) (đúng 4N bước) | O(N) |
| 4. Vật cản gần nhất | O(N) | O(N) × 4 mảng |
| 5. Lưới phẳng | O(N) | O(N) bytes |

In [16]:
rng = random.Random(2257)
def rand_case(m, n, pg, pw):
    cells = [(r, c) for r in range(m) for c in range(n)]; rng.shuffle(cells)
    k1 = max(1, int(pg * m * n)); k2 = max(1, int(pw * m * n))
    return [list(x) for x in cells[:k1]], [list(x) for x in cells[k1:k1 + k2]]
for m, n in [(316, 316), (1, 100000), (100000, 1)]:
    gs, ws = rand_case(m, n, 0.01, 0.05)
    res = []
    for name, f in [("simulate", simulate), ("four_sweeps", four_sweeps), ("nearest_blockers", nearest_blockers),
                    ("flat_simulate", flat_simulate), ("Solution", Solution().countUnguarded)]:
        t = time.perf_counter(); res.append(f(m, n, gs, ws)); dt = time.perf_counter() - t
        print(f"{m}x{n} {name:<17} {dt*1000:6.0f} ms")
    assert len(set(res)) == 1

316x316 simulate               8 ms
316x316 four_sweeps           31 ms
316x316 nearest_blockers      31 ms
316x316 flat_simulate          5 ms
316x316 Solution               8 ms


1x100000 simulate               5 ms


1x100000 four_sweeps           60 ms
1x100000 nearest_blockers      40 ms
1x100000 flat_simulate          3 ms
1x100000 Solution               4 ms
100000x1 simulate              49 ms


100000x1 four_sweeps          148 ms
100000x1 nearest_blockers     209 ms


100000x1 flat_simulate          4 ms
100000x1 Solution              22 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "simulate": simulate, "four_sweeps": four_sweeps, "nearest_blockers": nearest_blockers,
       "flat_simulate": flat_simulate, "Solution": Solution().countUnguarded}
edge = examples + [
    (1, 2, [[0, 0]], [[0, 1]], 0),                       # không còn ô trống
    (1, 5, [[0, 0]], [[0, 2]], 2),                       # G . W . .  -> 2 ô sau tường
    (1, 5, [[0, 2]], [[0, 4]], 0),                       # . . G . W
    (2, 2, [[0, 0]], [[1, 1]], 0),
    (3, 3, [[0, 0], [2, 2]], [[1, 1]], 0),
    (3, 4, [[1, 1]], [[1, 2]], 7),                       # tường chặn sang phải; cột 1 bị canh
    (5, 1, [[2, 0]], [[0, 0]], 0),                       # cột đơn: 1 trên (bị canh), 2 dưới (bị canh)
]
for m, n, gs, ws, exp in edge:
    for name, f in ALL.items():
        assert f(m, n, [list(x) for x in gs], [list(x) for x in ws]) == exp, (name, m, n, gs, ws, f(m, n, gs, ws), exp)
print("Edge case OK:", len(edge))

Edge case OK: 9


Kiểm tay case `(3, 4, G(1,1), W(1,2))`: hàng 1 là `. G W .` ⇒ chỉ `(1, 0)` bị canh theo phương ngang (tường chặn sang phải, nên `(1, 3)` không bị canh); cột 1: `(0, 1)` và `(2, 1)` bị canh. Có 12 − 2 = 10 ô trống, 3 ô bị canh ⇒ **7** ô không bị canh. Vẽ lại để đối chiếu:

In [18]:
print(Solution().countUnguarded(3, 4, [[1, 1]], [[1, 2]]), "| brute:", brute(3, 4, [[1, 1]], [[1, 2]]))
render(3, 4, [[1, 1]], [[1, 2]], seen=seen_cells(3, 4, [[1, 1]], [[1, 2]]))

7 | brute: 7
   . o . .
   o G W .
   . o . .


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [19]:
rng = random.Random(42)
for _ in range(800):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    if m * n < 2: continue
    cells = [[r, c] for r in range(m) for c in range(n)]; rng.shuffle(cells)
    g_cnt = rng.randint(1, max(1, m * n // 3)); w_cnt = rng.randint(1, max(1, m * n - g_cnt))
    w_cnt = min(w_cnt, m * n - g_cnt)
    if w_cnt < 1: continue
    gs, ws = cells[:g_cnt], cells[g_cnt:g_cnt + w_cnt]
    ref = brute(m, n, gs, ws)
    for name, f in ALL.items():
        assert f(m, n, gs, ws) == ref, (name, m, n, gs, ws)
print("800 case ngẫu nhiên OK")

800 case ngẫu nhiên OK


### 12.3 Tính chất: đặt thêm lính vào một ô trống không bao giờ làm tăng số ô không bị canh

Lính mới chặn tia của lính khác, nhưng phần phía sau lại do chính nó canh (xem Lỗi 4), nên không ô nào "hết bị canh". (Số ô không bị canh có thể **không đổi** nếu ô đặt lính vốn đã bị canh và lính mới không canh thêm ô nào.)

In [20]:
rng = random.Random(7)
for _ in range(300):
    m, n = rng.randint(2, 8), rng.randint(2, 8)
    cells = [[r, c] for r in range(m) for c in range(n)]; rng.shuffle(cells)
    gs, ws, rest = cells[:2], cells[2:5], cells[5:]
    if not rest: continue
    base = Solution().countUnguarded(m, n, gs, ws)
    new = Solution().countUnguarded(m, n, gs + [rest[0]], ws)
    assert new <= base                          # lính mới không làm ô nào "hết bị canh"
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tia dừng ở ô đã bị canh (coi `SEEN` là vật cản)

In [21]:
def wrong_stop_on_seen(m, n, guards, walls):
    g = [[0] * n for _ in range(m)]
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and g[x][y] == 0:      # SAI: dừng cả ở ô 3
                g[x][y] = 3; x += dr; y += dc
    return sum(row.count(0) for row in g)

args = (3, 3, [[0, 1], [1, 0]], [[2, 2]])
print("bản sai :", wrong_stop_on_seen(*args), "| bản đúng:", Solution().countUnguarded(*args))

bản sai : 1 | bản đúng: 0


**Vì sao sai:** lính `(0, 1)` đánh dấu cột 1 (gồm `(1, 1)`, `(2, 1)`); lính `(1, 0)` nhìn sang phải gặp `(1, 1)` đã `SEEN` và **dừng**, bỏ sót `(1, 2)`. Ô bị canh không chặn tầm nhìn. **Sửa:** chỉ dừng ở tường/lính.

### Lỗi 2 — Không đánh dấu mọi lính trước khi chiếu tia (dùng `set` các ô bị canh)

Một cách viết hay gặp: duyệt lính, coi lính **đã xử lý** là vật cản, gom ô bị canh vào `set`, cuối cùng lấy `m·n − tường − lính − |set|`. Lính **chưa xử lý** khi đó bị coi như ô trống ⇒ bị đưa vào `set` ⇒ trừ hai lần.

In [22]:
def wrong_set_count(m, n, guards, walls):
    walls_s = {tuple(w) for w in walls}; seen = set(); guard_done = set()
    for r, c in guards:
        guard_done.add((r, c))
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and (x, y) not in walls_s and (x, y) not in guard_done:
                seen.add((x, y)); x += dr; y += dc            # có thể "nhìn xuyên" lính chưa xử lý
    return m * n - len(walls) - len(guards) - len(seen)

args3 = (1, 3, [[0, 0], [0, 1]], [[0, 2]])
print("bản sai :", wrong_set_count(*args3), "| bản đúng:", Solution().countUnguarded(*args3))

bản sai : -1 | bản đúng: 0


Ở `args3`, lính `(0, 0)` "nhìn" ô `(0, 1)` (đang là lính chưa xử lý) và thêm nó vào `seen` ⇒ trừ hai lần ⇒ kết quả âm. **Sửa:** đánh dấu toàn bộ lính/tường **trước**, rồi mới chiếu tia.

### Lỗi 3 — Đếm cả ô lính/tường vào kết quả

In [23]:
def wrong_count_all(m, n, guards, walls):
    g = [[0] * n for _ in range(m)]
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and g[x][y] not in (1, 2):
                g[x][y] = 3; x += dr; y += dc
    return sum(1 for row in g for v in row if v != 3)            # SAI: tính cả 1 và 2

print("bản sai :", wrong_count_all(*examples[1][:4]), "| bản đúng:", Solution().countUnguarded(*examples[1][:4]))

bản sai : 9 | bản đúng: 4


**Sửa:** chỉ đếm ô có giá trị 0 (trống, không bị canh).

### Lỗi 4 — Nghĩ rằng phải xử lý "lính chặn tia lính khác" một cách đặc biệt

Bỏ điều kiện "lính chặn tia" (chỉ dừng ở tường) **vẫn cho cùng đáp án** — vì phần sau lính bị chặn vẫn bị chính lính đó canh. Nhưng nó làm mất tính khấu hao ⇒ chậm. Ta kiểm chứng cả hai điều:

In [24]:
def no_guard_block(m, n, guards, walls):
    g = [[0] * n for _ in range(m)]
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and g[x][y] != 1:     # chỉ dừng ở tường
                if g[x][y] == 0: g[x][y] = 3
                x += dr; y += dc
    return sum(row.count(0) for row in g)

rng = random.Random(4)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(2, 6)
    cells = [[r, c] for r in range(m) for c in range(n)]; rng.shuffle(cells)
    gs, ws = cells[:rng.randint(1, 3)], cells[3:3 + rng.randint(1, 3)]
    if not ws: continue
    assert no_guard_block(m, n, gs, ws) == Solution().countUnguarded(m, n, gs, ws)
row_guards = [[0, c] for c in range(0, 4000, 2)]
t = time.perf_counter(); no_guard_block(1, 4000, row_guards, [[0, 3999]]); t1 = time.perf_counter() - t
t = time.perf_counter(); Solution().countUnguarded(1, 4000, row_guards, [[0, 3999]]); t2 = time.perf_counter() - t
print(f"đáp án giống nhau; nhưng 2000 lính trên một hàng: không chặn {t1*1000:.0f} ms vs có chặn {t2*1000:.1f} ms")

đáp án giống nhau; nhưng 2000 lính trên một hàng: không chặn 770 ms vs có chặn 1.3 ms


**Bài học:** điều kiện "lính chặn tia" không ảnh hưởng tính đúng nhưng **quyết định độ phức tạp** (O(G·(m+n)) → O(m·n)).

### Lỗi 5 — Dùng lưới `m × n` với `m`, `n` tới 10⁵ mà nghĩ sẽ tràn bộ nhớ

Không tràn: đề đảm bảo `m·n ≤ 10⁵`. Lỗi thật thường gặp là tạo lưới **`[[0] * n] * m`** (các hàng là **cùng một list**):

In [25]:
def wrong_shared_rows(m, n, guards, walls):
    g = [[0] * n] * m                                     # SAI: m tham chiếu tới cùng một hàng
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and g[x][y] not in (1, 2):
                g[x][y] = 3; x += dr; y += dc
    return sum(row.count(0) for row in g)

print("bản sai :", wrong_shared_rows(*examples[0][:4]), "| bản đúng:", Solution().countUnguarded(*examples[0][:4]))

bản sai : 4 | bản đúng: 7


**Sửa:** `[[0] * n for _ in range(m)]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhìn theo 4 hướng đến khi bị chặn" ⇒ mô phỏng tia, **dừng ở vật cản** để có khấu hao O(m·n); hoặc 4 lượt quét với cờ.
- Đánh dấu **mọi vật cản trước**, rồi mới mô phỏng.
- Phân biệt "vật cản" (tường, lính) với "trạng thái" (đã bị canh) — trạng thái không chặn tia.
- `m·n ≤ 10⁵` dù từng chiều tới 10⁵ ⇒ lưới vẫn nhỏ; nhớ `[[0]*n for _ in range(m)]`.
- Bài cùng họ: 2201, 361 Bomb Enemy (4 lượt quét / vật cản gần nhất), 1222 Queens That Can Attack the King, 999 Available Captures for Rook.

## 15. Biến thể của bài

### 15.1 Biến thể A — Lính nhìn cả **8 hướng** (thêm đường chéo)

In [26]:
def count_unguarded_8(m, n, guards, walls):
    g = [[0] * n for _ in range(m)]
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    dirs = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
    for r, c in guards:
        for dr, dc in dirs:
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and g[x][y] not in (1, 2):
                g[x][y] = 3; x += dr; y += dc
    return sum(row.count(0) for row in g)

print(count_unguarded_8(*examples[0][:4]), "(4 hướng:", Solution().countUnguarded(*examples[0][:4]), ")")

4 (4 hướng: 7 )


### 15.2 Biến thể B — Đếm **số lính** canh mỗi ô, trả số ô bị ≥ k lính canh

Dùng cách 4 (vật cản gần nhất): mỗi hướng góp tối đa 1 lính ⇒ số lính canh ô = số hướng có vật cản gần nhất là lính.

In [27]:
def cells_guarded_by_at_least(m, n, guards, walls, k):
    grid = [[0] * n for _ in range(m)]
    for r, c in walls: grid[r][c] = 1
    for r, c in guards: grid[r][c] = 2
    cnt = [[0] * n for _ in range(m)]
    def sweep(cells):
        last = 0
        for r, c in cells:
            if grid[r][c]: last = grid[r][c]
            elif last == 2: cnt[r][c] += 1
    for r in range(m):
        sweep([(r, c) for c in range(n)]); sweep([(r, c) for c in range(n - 1, -1, -1)])
    for c in range(n):
        sweep([(r, c) for r in range(m)]); sweep([(r, c) for r in range(m - 1, -1, -1)])
    return sum(1 for r in range(m) for c in range(n) if grid[r][c] == 0 and cnt[r][c] >= k)

m, n, gs, ws, _ = examples[0]
print([cells_guarded_by_at_least(m, n, gs, ws, k) for k in range(1, 5)])

[11, 2, 0, 0]


### 15.3 Biến thể C — Lính chỉ nhìn xa tối đa `d` ô

In [28]:
def count_unguarded_range(m, n, guards, walls, d):
    g = [[0] * n for _ in range(m)]
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y, steps = r + dr, c + dc, 0
            while steps < d and 0 <= x < m and 0 <= y < n and g[x][y] not in (1, 2):
                g[x][y] = 3; x += dr; y += dc; steps += 1
    return sum(row.count(0) for row in g)

print([count_unguarded_range(*examples[0][:4], d) for d in range(0, 6)], "-> hội tụ về", Solution().countUnguarded(*examples[0][:4]))

[18, 12, 8, 7, 7, 7] -> hội tụ về 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từ ô nhìn ra | tìm vật cản đầu tiên mỗi hướng | O(N(m+n)) | O(N) | kiểm chứng |
| 2. Mô phỏng tia | lính chiếu tia, dừng ở vật cản | O(N) khấu hao | O(N) | **mặc định**, trực quan |
| 3. Bốn lượt quét | cờ "đang nhìn" | O(4N) | O(N) | dễ chứng minh, không while lồng |
| 4. Vật cản gần nhất | tách tính toán và quyết định | O(N) | O(4N) | biến thể đếm số lính |
| 5. Lưới phẳng | `bytearray`, `count` trong C | O(N) | O(N) bytes | tối ưu hằng số |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countUnguarded(2, 7, [[1,5],[1,1],[1,6],[0,2]], [[0,6],[0,3],[0,5]])`?

**Bài 2 (tính tay):** Với `m = 1, n = 8`, lính ở cột `1, 5`, tường ở cột `3`: ô nào không bị canh?

**Bài 3 (điền TODO):** Viết `bt3_todo(m, n, guards, walls)` trả về **danh sách tọa độ** ô trống không bị canh (theo thứ tự hàng rồi cột).

**Bài 4 (tự viết lại):** Viết `bt4_todo(m, n, guards, walls)` cài cách 3 (bốn lượt quét) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(m, n, guards, walls)` trả về **số lính tối thiểu cần thêm** (đặt vào ô trống) để mọi ô trống đều bị canh — với lưới nhỏ (`m·n ≤ 12`) bằng vét cạn tập con.

In [29]:
def bt3_todo(m, n, guards, walls):
    # TODO: danh sách ô trống không bị canh
    pass

def bt4_todo(m, n, guards, walls):
    # TODO: bốn lượt quét
    pass

def bt5_todo(m, n, guards, walls):
    # TODO: số lính tối thiểu cần thêm (vét cạn, lưới nhỏ)
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: vẽ lưới 2 × 7. Mức 2: hàng 1 toàn lính/bị canh; hàng 0 có tường ở cột 3, 5, 6 và lính ở cột 2. Mức 3: hàng 1 mọi ô trống đều bị lính hàng 1 canh; ở hàng 0, lính `(0, 2)` canh `(0, 0), (0, 1)`. Ô `(0, 4)` bị kẹp giữa tường cột 3 và cột 5, phía dưới là `(1, 4)` (trống — không có lính trong cột 4) ⇒ không ai nhìn thấy ⇒ đáp án `1`. Chạy `render` + `seen_cells` để xác nhận.

**Bài 2.** Mức 1: `. G . W . G . .`. Mức 2: lính 1 canh cột 0, 2; lính 5 canh cột 4, 6, 7. Mức 3: không ô nào ⇒ `[]`, đáp án 0.

**Bài 3.** Mức 1: như `simulate`. Mức 2: duyệt `r, c` và lấy các ô 0. Mức 3: `[[r, c] for r ... for c ... if g[r][c] == 0]`.

**Bài 4.** Mức 1: hàm `sweep(cells)`. Mức 2: 2 lượt mỗi hàng, 2 lượt mỗi cột. Mức 3: đếm ô trống chưa đánh dấu.

**Bài 5.** Mức 1: thử thêm 0, 1, 2… lính. Mức 2: với mỗi tập ô trống kích thước `k`, gọi `Solution` với lính mới. Mức 3: dừng ở `k` đầu tiên cho kết quả 0.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `1` (chỉ ô `(0, 4)`), in ra và assert với brute ở cell dưới. **Bài 2:** `[]` (0 ô).

In [30]:
from itertools import combinations
a1 = Solution().countUnguarded(2, 7, [[1, 5], [1, 1], [1, 6], [0, 2]], [[0, 6], [0, 3], [0, 5]])
assert a1 == brute(2, 7, [[1, 5], [1, 1], [1, 6], [0, 2]], [[0, 6], [0, 3], [0, 5]])
print("Bài 1:", a1)
assert Solution().countUnguarded(1, 8, [[0, 1], [0, 5]], [[0, 3]]) == 0

def bt3(m, n, guards, walls):
    g = [[0] * n for _ in range(m)]
    for r, c in walls: g[r][c] = 1
    for r, c in guards: g[r][c] = 2
    for r, c in guards:
        for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            x, y = r + dr, c + dc
            while 0 <= x < m and 0 <= y < n and g[x][y] not in (1, 2):
                g[x][y] = 3; x += dr; y += dc
    return [[r, c] for r in range(m) for c in range(n) if g[r][c] == 0]

def bt4(m, n, guards, walls):
    return four_sweeps(m, n, guards, walls)

def bt5(m, n, guards, walls):
    occupied = {tuple(x) for x in guards} | {tuple(x) for x in walls}
    empties = [[r, c] for r in range(m) for c in range(n) if (r, c) not in occupied]
    for k in range(len(empties) + 1):
        for extra in combinations(empties, k):
            if Solution().countUnguarded(m, n, guards + list(extra), walls) == 0:
                return k
    return len(empties)

rng = random.Random(1957)
for _ in range(300):
    m, n = rng.randint(1, 4), rng.randint(2, 3)
    cells = [[r, c] for r in range(m) for c in range(n)]; rng.shuffle(cells)
    gs, ws = cells[:1], cells[1:1 + rng.randint(1, 2)]
    lst = bt3(m, n, gs, ws)
    assert len(lst) == brute(m, n, gs, ws) and bt4(m, n, gs, ws) == brute(m, n, gs, ws)
    k = bt5(m, n, gs, ws)
    assert (k == 0) == (brute(m, n, gs, ws) == 0)
print("bt5 ví dụ 2 (lính bị bao vây):", bt5(*examples[1][:4]))
print("Bài 1 - ô không bị canh:", bt3(2, 7, [[1, 5], [1, 1], [1, 6], [0, 2]], [[0, 6], [0, 3], [0, 5]]))
print("Đáp án bài tập: tất cả assert OK")

Bài 1: 1
bt5 ví dụ 2 (lính bị bao vây): 4
Bài 1 - ô không bị canh: [[0, 4]]
Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: ở ví dụ 2, mỗi góc chỉ "thông" với các ô kề nó là tường ⇒ không ô trống nào khác nhìn thấy nó; đặt lính lên một góc thì góc đó không còn là ô trống. Vì vậy cần đặt lính lên đủ 4 góc — cell trên in ra đúng con số này.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
g = [[0]*n for _ in range(m)]            # 0 trống, 1 tường, 2 lính, 3 bị canh
for r, c in walls:  g[r][c] = 1           # đánh dấu MỌI vật cản trước
for r, c in guards: g[r][c] = 2
for r, c in guards:
    for dr, dc in ((0,1),(0,-1),(1,0),(-1,0)):
        x, y = r + dr, c + dc
        while 0 <= x < m and 0 <= y < n and g[x][y] not in (1, 2):   # ô 3 không chặn
            g[x][y] = 3; x += dr; y += dc
return sum(row.count(0) for row in g)
```

**Checklist:**
- [ ] Đánh dấu tường **và** lính trước khi chiếu tia.
- [ ] Tia dừng ở tường/lính, **không** dừng ở ô đã bị canh.
- [ ] Chỉ đếm ô 0.
- [ ] Lưới tạo bằng list comprehension (không `[[0]*n]*m`).
- [ ] Đã thử: lưới 1 hàng/1 cột, lính bị bao vây, nhiều lính cùng hàng.